# Handling NaN, Inf and Missing Numerical Values

## Definition
In numerical computing, special floating-point values represent edge cases:

| Value | Meaning | How it arises |
|-------|---------|---------------|
| `np.nan` | Not a Number | 0/0, sqrt(-1), inf - inf |
| `np.inf` | Positive Infinity | x/0 for x>0, overflow |
| `-np.inf` | Negative Infinity | x/0 for x<0 |

### Key properties
- `nan != nan` is **True** (nan is not equal to anything, including itself!)
- `nan` propagates through arithmetic: `nan + 5 = nan`
- Comparisons with nan always return `False`
- NaN only exists for **float** dtypes (not int)

### NaN-safe functions
NumPy has `nan*` versions of common aggregations:
`nansum`, `nanmean`, `nanstd`, `nanvar`, `nanmin`, `nanmax`, `nanmedian`, `nanpercentile`, `nanprod`

In [ ]:
import numpy as np
import warnings

# Creating NaN and Inf
nan  = np.nan
inf  = np.inf
ninf = -np.inf

print('nan:', nan)
print('inf:', inf)
print('-inf:', ninf)
print()

# Operations that produce NaN and Inf
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    print('0.0 / 0.0:', 0.0 / 0.0)    # nan
    print('1.0 / 0.0:', 1.0 / 0.0)    # inf
    print('-1.0 / 0.0:', -1.0 / 0.0)  # -inf
    print('np.sqrt(-1):', np.sqrt(-1.0))  # nan
    print('np.log(0):', np.log(0.0))      # -inf

In [ ]:
# Detecting NaN and Inf
arr = np.array([1.0, np.nan, np.inf, -np.inf, 2.0, np.nan])

print('isnan:', np.isnan(arr))
print('isinf:', np.isinf(arr))
print('isfinite:', np.isfinite(arr))

print('Any nan?', np.any(np.isnan(arr)))
print('Count of nan:', np.sum(np.isnan(arr)))

In [ ]:
# NaN propagation
arr = np.array([1.0, 2.0, np.nan, 4.0, 5.0])
print('sum:', np.sum(arr))      # nan — propagates!
print('mean:', np.mean(arr))    # nan
print('max:', np.max(arr))      # nan

# NaN-safe alternatives
print('nansum:', np.nansum(arr))    # 12.0
print('nanmean:', np.nanmean(arr))  # 3.0
print('nanmax:', np.nanmax(arr))    # 5.0

In [ ]:
# Replacing NaN values
arr = np.array([1.0, np.nan, 3.0, np.nan, 5.0])

# Replace NaN with 0
arr_filled_0 = np.where(np.isnan(arr), 0, arr)
print('NaN -> 0:', arr_filled_0)

# Replace NaN with mean (imputation)
mean_val = np.nanmean(arr)
arr_imputed = np.where(np.isnan(arr), mean_val, arr)
print('NaN -> mean:', arr_imputed)

# Using np.nan_to_num
clean = np.nan_to_num(arr, nan=0.0, posinf=1e9, neginf=-1e9)
print('nan_to_num:', clean)

In [ ]:
# NaN in 2D — finding and filling
mat = np.array([[1.0, np.nan, 3.0],
                [np.nan, 5.0, 6.0],
                [7.0, 8.0, np.nan]])

print('NaN locations:\n', np.isnan(mat))
print('Row means (ignoring NaN):', np.nanmean(mat, axis=1))

# Column-wise mean imputation
col_means = np.nanmean(mat, axis=0)
print('Column means:', col_means)

# Fill NaN with column mean
mat_filled = mat.copy()
nan_rows, nan_cols = np.where(np.isnan(mat_filled))
mat_filled[nan_rows, nan_cols] = col_means[nan_cols]
print('After column imputation:\n', mat_filled)

In [ ]:
# Inf handling
arr = np.array([1.0, np.inf, -np.inf, 2.0, 3.0])
print('Original:', arr)

# Replace inf with large finite value
clean = np.nan_to_num(arr, posinf=1e10, neginf=-1e10)
print('After nan_to_num:', clean)

# Check sign of inf
print('inf > 1e300:', np.inf > 1e300)   # True
print('-inf < 1e300:', -np.inf < 1e300) # True

In [ ]:
# EXCEPTION: integer arrays cannot hold NaN
try:
    arr = np.array([1, 2, np.nan], dtype=np.int32)
except ValueError as e:
    print('ValueError:', e)

# Only float arrays can hold NaN
arr = np.array([1, 2, np.nan])  # auto-promotes to float64
print('dtype:', arr.dtype, '  values:', arr)